# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW_CREDIT_RECORD= Path("..") / "data" / "raw" / "credit_record.csv"          # PREENCHER: nome do arquivo
RAW_APPLICATION_RECORD= Path("..") / "data" / "raw" / "application_record.csv"     # PREENCHER: nome do arquivo
PROCESSED = Path("..") / "data" / "processed"
TARGET = "TARGET"                                            # PREENCHER: variável alvo

pd.set_option("display.max_columns", None)

In [2]:
# Criando cópias para não modificar o DATASET principal
df_credit_record = pd.read_csv(RAW_CREDIT_RECORD)
df_application_record = pd.read_csv(RAW_APPLICATION_RECORD)

## 1. Dados faltantes

### Tratamento de valores ausentes em `OCCUPATION_TYPE`

**Justificativa/Decisão:** A coluna possui uma quantidade expressiva de dados ausentes (134.177 linhas), em sua grande maioria associados a aposentados (`Pensioner`). Como excluir essas linhas causaria perda severa de dados válidos, optou-se por preenchê-los com a categoria textual `"Unknown"`, transformando a ausência em uma informação categórica útil para o modelo.

In [3]:
print("Quantidade de nulos antes do tratamento: ", df_application_record["OCCUPATION_TYPE"].isna().sum())


df_application_record["OCCUPATION_TYPE"] = df_application_record["OCCUPATION_TYPE"].fillna("Unknown")

print("Quantidade de nulos pós do tratamento: ", df_application_record["OCCUPATION_TYPE"].isna().sum())

Quantidade de nulos antes do tratamento:  134203
Quantidade de nulos pós do tratamento:  0


### Tratamento de IDs duplicados em `application_record`
**Justificativa:** Foi identificada a presença de 47 IDs duplicados na base `application_record` contendo informações cadastrais conflitantes.  

Como o `ID`  serve como chave de união com a variável alvo, manter essas duplicatas geraria  inconsistências. Removeu-se todas as linhas associadas a esses IDs conflitantes para preservar a unicidade do cadastro.

In [4]:
print("Formato da base antes tratamento:", df_application_record.shape)
# Identificar IDs duplicados
ids_duplicados_df_application_record= df_application_record.loc[df_application_record["ID"].duplicated(keep=False), "ID"].unique()


# Remover todos os registros com IDs duplicados
df_application_record = df_application_record[~df_application_record["ID"].isin(ids_duplicados_df_application_record)].copy()
print("Formato da base após tratamento:", df_application_record.shape)

Formato da base antes tratamento: (438557, 18)
Formato da base após tratamento: (438463, 18)


## 2. Definição da variável alvo
_Se houve binarização, qual limiar e por quê? Justifique com base na distribuição, não por convenção._

**Justificativa da binarização:**  
A variável original `STATUS` possui 8 categorias ordinais/textuais. Para transformá-la em classificação binária, o limiar escolhido foi de atrasos $\ge$ 30 dias (`STATUS` 1, 2, 3, 4 e 5) definidos como **Mau Pagador (1)** conforme regra definida pela equipe de análise de dados..  
Os meses com atrasos leves, quitados ou sem empréstimos foram mapeados como **Bom Pagador (0)**. No nível do cliente, considerou-se o pior caso histórico (máximo). A distribuição final mostra forte desbalanceamento (11,77% de maus pagadores na base fundida).


In [5]:
# Criar indicador de mês com atraso grave
df_credit_record["BAD_MONTH"] = df_credit_record["STATUS"].isin(["1","2","3", "4", "5"]).astype(int)

# Consolidar a variável alvo por cliente (ID)
target = df_credit_record.groupby("ID")["BAD_MONTH"].max().reset_index()
target = target.rename(columns={"BAD_MONTH": "TARGET"})


### Junção das bases `application_record` e `target`

**Justificativa:** Nem todos os clientes cadastrados possuem histórico de crédito e vice-versa. Utilizou-se um `merge` do tipo `inner` (interseção) para garantir que a base final contenha estritamente clientes que possuem atributos preditivos e rótulo de destino associado, resultando em 36.457 registros únicos.


In [6]:
df = df_application_record.merge(target, on="ID", how="inner")# Fazendo INNER JOIN 
df.shape# verificando o tamanho do dataset após o Inner Join

(36457, 19)

## 3. Normalização / padronização

**Escolha e justificativa:** *Nenhum neste momento.* Conforme instruído pelo 
template de arquitetura, técnicas como `StandardScaler` ou `MinMaxScaler` devem 
ser aplicadas estritamente dentro de um `Pipeline` de Machine Learning na etapa 
de treinamento (Notebook 03). Executar o ajuste do scaler antes do split de 
treino/teste causaria vazamento de informação (*data leakage*).

## 4. Feature engineering


### Justificativa das transformações e codificações sequenciais:
1. **Interpretabilidade e Saneamento Numérico:** `AGE_YEARS` e `EMPLOYED_YEARS` convertem os registros regressivos em anos positivos. O valor técnico anômalo `365243` foi isolado em `EMPLOYED_SPECIAL_VALUE` e tratado com `-1` para não enviesar modelos lineares ou de distância.
2. **Suavização de Assimetria:** Aplicou-se a transformação Logarítmica (`log1p`) em `AMT_INCOME_TOTAL`, gerando `LOG_INCOME` para estabilizar a variância da renda.
3. **Limpeza de Constantes:** A coluna `FLAG_MOBIL` foi descartada por possuir variância zero (todos os registros iguais a 1).
4. **Codificação Categórica e Unificação de Escopo:** Para garantir que a base salva em disco contenha a estrutura final de modelagem, as variáveis de texto foram processadas sequencialmente pelo `ColumnTransformer` (OrdinalEncoder para escolaridade e OneHotEncoder para as demais categóricas) com remoção automática de prefixos técnicos, consolidando todas as 44' colunas diretamente na base tratada final.


In [7]:
# 1. Idade interpretável em anos
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365).round(1)

# 2. Tratamento do valor anômalo de emprego
df["EMPLOYED_SPECIAL_VALUE"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED_TREATED"] = df["DAYS_EMPLOYED"].replace(365243, pd.NA)
df["EMPLOYED_YEARS"] = (-pd.to_numeric(df["DAYS_EMPLOYED_TREATED"], errors="coerce") / 365).round(1)
df["EMPLOYED_YEARS"] = df["EMPLOYED_YEARS"].fillna(-1)

# 3. Transformação logarítmica da renda para suavizar assimetria
df["LOG_INCOME"] = np.log1p(df["AMT_INCOME_TOTAL"])

# 4. Mapeamento de variáveis categóricas binárias Y/N para 1/0
df["FLAG_OWN_CAR"] = np.where(df["FLAG_OWN_CAR"].isin(["Y", 1, "1"]), 1, 0)
df["FLAG_OWN_REALTY"] = np.where(df["FLAG_OWN_REALTY"].isin(["Y", 1, "1"]), 1, 0)
df["CODE_GENDER"] = np.where(df["CODE_GENDER"].isin(["M", 1, "1"]), 1, 0)

# Remoção de colunas originais redundantes e colunas sem variância (FLAG_MOBIL)
df_application_record_tratado = df.drop(
    columns=[
        "ID",
        "AMT_INCOME_TOTAL",
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_EMPLOYED_TREATED",
        "CNT_CHILDREN",
        "FLAG_MOBIL"
    ]
)

# 

In [8]:
# Verificação rápida da primeira fase do tratamento
df_application_record_tratado.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET,AGE_YEARS,EMPLOYED_SPECIAL_VALUE,EMPLOYED_YEARS,LOG_INCOME
0,1,1,1,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,2.0,1,32.9,0,12.4,12.965712
1,1,1,1,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,2.0,1,32.9,0,12.4,12.965712
2,1,1,1,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Security staff,2.0,0,58.8,0,3.1,11.630717
3,0,0,1,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0,0,52.4,0,8.4,12.506181
4,0,0,1,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0,0,52.4,0,8.4,12.506181


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# 1. Definição da ordem hierárquica manual para a Escolaridade (Ordinal)
educacao_ordem = [
    "Lower secondary",
    "Secondary / secondary special",
    "Incomplete higher",
    "Higher education",
    "Academic degree"
]

# 2. Separação das colunas por estratégia
colunas_onehot = ["NAME_INCOME_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE"]
coluna_ordinal = ["NAME_EDUCATION_TYPE"]

# 3. Construção do Transformador de Colunas sem gerar prefixos
precomputador_categorico = ColumnTransformer(
    transformers=[
        ("escolaridade_ordinal", OrdinalEncoder(categories=[educacao_ordem]), coluna_ordinal),
        ("categoricas_nominais", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), colunas_onehot)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)


In [10]:
# 1. Aplica o transformador e recupera os nomes puros das colunas
dados_transformados = precomputador_categorico.fit_transform(df_application_record_tratado)
nomes_colunas_limpas = precomputador_categorico.get_feature_names_out()

# 2. Sobrescreve a variável com o resultado final totalmente tratado
df_application_record_tratado = pd.DataFrame(dados_transformados, columns=nomes_colunas_limpas)

# 3. Exibição de validação
print("="*80)
print(f"📐 Formato final da tabela em 'df_application_record_tratado': {df_application_record_tratado.shape}")
print("="*80)
display(df_application_record_tratado.head())



📐 Formato final da tabela em 'df_application_record_tratado': (36457, 44)


,NAME_EDUCATION_TYPE,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Working,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,OCCUPATION_TYPE_Waiters/barmen staff,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,TARGET,AGE_YEARS,EMPLOYED_SPECIAL_VALUE,EMPLOYED_YEARS,LOG_INCOME
0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,2.0,1.0,32.9,0.0,12.4,12.965712
1,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,2.0,1.0,32.9,0.0,12.4,12.965712
2,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,2.0,0.0,58.8,0.0,3.1,11.630717
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,52.4,0.0,8.4,12.506181
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,52.4,0.0,8.4,12.506181


## 5. Salvar dataset tratado

In [11]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df_application_record_tratado.to_csv(PROCESSED / "df_application_record_tratado.csv", index=False)

print(f"✅ Pipeline concluído. Dataset salvo em: {PROCESSED / 'df_application_record_tratado.csv'}")
print(f"Formato final dos dados salvos: {df_application_record_tratado.shape}")



✅ Pipeline concluído. Dataset salvo em: ../data/processed/df_application_record_tratado.csv
Formato final dos dados salvos: (36457, 44)
